# Phase 8 — Multi-Flow Event Correlation & Decision Support Compression

**Status:** ✅ Completed (pipeline v1.0)  
**Engine Implementation:** `ml_service/src/correlation/correlator.py`  
**Flag Generator:** `ml_service/src/correlation/generate_flags.py`  
**Evaluation Suite:** `ml_service/src/correlation/evaluate_correlation.py`  
**Pre-Registration Baseline:** `artifacts/correlation/preregistration_v1.md`  
**Evaluation Report:** `artifacts/correlation/correlation_report_v1.md`  
**Telemetry Summary:** `artifacts/correlation/incident_summary_v1.json`  

---

## Executive Summary & Research Methodology
Phase 8 implements multi-flow sessionization to group individual network intrusion alerts into coherent, actionable security incidents. Operating downstream of the Phase 5 XGBoost classifier and Phase 7 Security Rule Engine, it compresses high-volume telemetry without sacrificing attack attribution:
- **Strict Ground-Truth Blindness:** The correlation engine (`correlator.py`) ingests only `[row_id, source_ip, destination_ip, destination_port, epoch_s, predicted_class, confidence]`. Telemetry containing `label` or `source_day` is rejected programmatically.
- **Tuesday Patator Acceptance Gate:** Evaluated on the unseen 20% test partition, the Tuesday brute-force capture served as the mandatory pre-registered gateway. All 5 pre-registered acceptance criteria passed unconditionally (FTP fragmentation = 2, SSH fragmentation = 1, Purity = 100.0%, Coverage = 99.95%).
- **Massive Alert Fatigue Reduction:** Compressed **86,170 raw alert flows** down to **3,296 incidents** on the primary test partition (**96.17% alert reduction**, **26.1x compression factor**) while maintaining **96.41% flow-weighted purity**.
- **PortScan Key Ablation ($K_3$ vs $K_2$):** Relaxing destination port from the grouping key collapses horizontal port-sweep fragments by **32.9x** (from 2,172 incidents to 66 incidents) with zero loss of threat attribution.
- **Empirical Merge Verification:** Pre-registered cross-threat campaign merge predictions were validated against empirical observations.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import display  # type: ignore
except ImportError:
    display = print

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

# Load incident summary artifact
summary_path = Paths.ARTIFACTS_DIR / 'correlation' / 'incident_summary_v1.json'
with open(summary_path, 'r', encoding='utf-8') as f:
    summary = json.load(f)

tues_gate = summary['tuesday_gate']
test_primary = summary['test_split_primary']
all_test = test_primary['ALL_TEST']

print('Successfully loaded Phase 8 Event Correlation Summary.')
print(f"Primary Test Partition: {all_test['n_raw_flagged']:,} raw alerts -> {all_test['n_incidents']:,} incidents")
print(f"Alert Load Reduction: {all_test['reduction_ratio']*100:.2f}%")
print(f"Flow-Weighted Incident Purity: {all_test['metrics']['flow_weighted_purity']*100:.2f}%")

## 1. Tuesday Brute Force Acceptance Gate Verification
Per AGENTS.md, Tuesday Patator brute-force validation is the mandatory gateway before any generalized multi-day claims can be accepted. Below is the audited pass/fail scorecard against pre-registered criteria:

In [ ]:
gate_crit = tues_gate['criteria']
rows = []
labels_map = {
    'ftp_fragmentation': 'FTP-Patator Episode Fragmentation',
    'ssh_fragmentation': 'SSH-Patator Episode Fragmentation',
    'flow_weighted_purity': 'Flow-Weighted Incident Purity',
    'incident_weighted_purity': 'Incident-Weighted Incident Purity',
    'label_consistent_multi_flow_coverage': 'Label-Consistent Multi-Flow Coverage'
}

for k, v in gate_crit.items():
    meas = f"{v['measured']:.3f}%" if isinstance(v['measured'], float) else str(v['measured'])
    rows.append({
        'Criterion': labels_map.get(k, k),
        'Pre-Registered Goal': v['goal'],
        'Measured Result': meas,
        'Gate Verdict': 'PASS' if v['pass'] else 'FAIL'
    })

gate_df = pd.DataFrame(rows)
display(gate_df)

# Bar chart comparison of Tuesday compression
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['Raw Flagged Flows', 'Correlated Incidents'], [tues_gate['n_raw_flagged'], tues_gate['n_incidents']], color=['#e74c3c', '#2ecc71'])
for b in bars:
    yval = b.get_height()
    ax.text(b.get_x() + b.get_width()/2.0, yval + 30, f'{yval:,}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Count')
ax.set_title(f"Tuesday Patator Compression: {tues_gate['reduction_ratio']*100:.1f}% Reduction ({tues_gate['compression_factor']:.1f}x)", fontsize=12, fontweight='bold')
ax.set_ylim(0, tues_gate['n_raw_flagged'] * 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 2. Multi-Day Performance Breakdown: Test Partition
Evaluating the correlation engine across all 8 capture files on the unseen test partition ($N=504,160$):

In [ ]:
day_rows = []
for d, data in test_primary.items():
    if d == 'ALL_TEST':
        continue
    m = data['metrics']
    day_rows.append({
        'Capture File': d,
        'Raw Alerts': data['n_raw_flagged'],
        'Incidents': data['n_incidents'],
        'Multi-Flow (N>=2)': m['multi_flow_incidents'],
        'Singletons (N=1)': m['singletons'],
        'Reduction Ratio': f"{data['reduction_ratio']*100:.2f}%",
        'Flow Purity': f"{m['flow_weighted_purity']*100:.2f}%",
        'Consistent Coverage': f"{m['label_consistent_multi_flow_coverage_flagged']*100:.2f}%"
    })

perf_df = pd.DataFrame(day_rows)
display(perf_df)

# Visualizing Alert Reduction across days
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(perf_df))
width = 0.35

ax.bar(x - width/2, perf_df['Raw Alerts'], width, label='Raw Flagged Alerts', color='#34495e')
ax.bar(x + width/2, perf_df['Incidents'], width, label='Correlated Incidents', color='#3498db')

ax.set_ylabel('Count (Log Scale)')
ax.set_yscale('log')
ax.set_title('Raw Alerts vs. Correlated Incidents per Attack Capture (Log Scale)', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([name.replace('_', ' ') for name in perf_df['Capture File']], rotation=30, ha='right', fontsize=9)
ax.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Gap Window Sensitivity Sweep ($G \in \{60, 300, 900, 1800\}\text{ s}$)
Analyzing the trade-off between incident compression and attribution purity as the inactivity timeout $G$ increases:

In [ ]:
gap_sw = summary['gap_sweep']
gap_rows = []
for gk in ['G_60s', 'G_300s', 'G_900s', 'G_1800s']:
    g_data = gap_sw[gk]
    gm = g_data['metrics']
    gap_rows.append({
        'Inactivity Gap (G)': gk.replace('G_', '').replace('s', ' s'),
        'Total Incidents': g_data['n_incidents'],
        'Multi-Flow (N>=2)': gm['multi_flow_incidents'],
        'Singletons (N=1)': gm['singletons'],
        'Reduction Ratio': f"{g_data['reduction_ratio']*100:.2f}%",
        'Flow Purity': f"{gm['flow_weighted_purity']*100:.2f}%",
        'Consistent Coverage': f"{gm['label_consistent_multi_flow_coverage_flagged']*100:.2f}%"
    })

gap_df = pd.DataFrame(gap_rows)
display(gap_df)

# Plotting Gap Window Trade-offs
fig, ax1 = plt.subplots(figsize=(8, 4.5))
gaps = [60, 300, 900, 1800]
inc_counts = [gap_sw[f'G_{g}s']['n_incidents'] for g in gaps]
purities = [gap_sw[f'G_{g}s']['metrics']['flow_weighted_purity']*100 for g in gaps]

color = '#2980b9'
ax1.set_xlabel('Gap Window Timeout G (seconds)', fontweight='bold')
ax1.set_ylabel('Total Incidents Formed', color=color, fontweight='bold')
ax1.plot(gaps, inc_counts, marker='o', color=color, linewidth=2.5, label='Incident Count')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#27ae60'
ax2.set_ylabel('Flow-Weighted Purity (%)', color=color, fontweight='bold')
ax2.plot(gaps, purities, marker='s', linestyle='--', color=color, linewidth=2.5, label='Purity (%)')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(90, 101)

plt.title('Gap Window Timeout G Sensitivity: Incidents vs. Purity', fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Key Ablation Study: $K_3$ vs $K_2$ PortScan Collapse
Evaluating the impact of relaxing `Destination Port` from the tuple key for distributed scans:

In [ ]:
key_ab = summary['key_ablation']
ps_comp = key_ab['portscan_comparison']

key_rows = []
for kn, kd in [('K3_primary', 'K3: (Src IP, Dst IP, Dst Port)'), 
              ('K2_portscan_ablation', 'K2: (Src IP, Dst IP)'),
              ('K4_class_qualified', 'K4: (Src IP, Dst IP, Dst Port, Class)'),
              ('dest_port_only_baseline', 'Trivial: (Dst Port,)')]:
    k_data = key_ab[kn]
    km = k_data['metrics']
    key_rows.append({
        'Grouping Key Scheme': kd,
        'Incidents Formed': k_data['n_incidents'],
        'Reduction Ratio': f"{k_data['reduction_ratio']*100:.2f}%",
        'Flow Purity': f"{km['flow_weighted_purity']*100:.2f}%",
        'Consistent Coverage': f"{km['label_consistent_multi_flow_coverage_flagged']*100:.2f}%"
    })

key_df = pd.DataFrame(key_rows)
display(key_df)

# PortScan specific collapse visualization
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['K3 Primary (Port-Segregated)', 'K2 Ablation (Port-Agile)'], [ps_comp['K3_incidents'], ps_comp['K2_incidents']], color=['#e67e22', '#16a085'])
for b in bars:
    yval = b.get_height()
    ax.text(b.get_x() + b.get_width()/2.0, yval + 40, f'{yval:,}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Incident Count')
ax.set_title(f"PortScan Key Collapse: {ps_comp['K2_collapse_ratio']:.1f}x Incident Reduction", fontsize=12, fontweight='bold')
ax.set_ylim(0, ps_comp['K3_incidents'] * 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Architectural Comparison & Ingestion Ablations
Contrasting our dynamic sliding gap sessionizer against fixed tumbling windows, and inspecting hybrid ML $\cup$ Rule R104 alert ingestion:

In [ ]:
base_comp = summary['baseline_comparison']
ing_ab = summary['ingestion_ablation']

print('=== Architectural Baseline Comparison ===')
print(f"Sliding Gap Sessionizer (Ours): {base_comp['sliding_gap_300s']['n_incidents']:,} incidents | Reduction: {base_comp['sliding_gap_300s']['reduction_ratio']*100:.2f}%")
print(f"Fixed Tumbling Window (300s):   {base_comp['fixed_tumbling_300s']['n_incidents']:,} incidents | Reduction: {base_comp['fixed_tumbling_300s']['reduction_ratio']*100:.2f}%")
print('\n=== Ingestion Ablation (ML-only vs Hybrid) ===')
print(f"ML-Only Ingestion:            {ing_ab['ml_only']['n_raw_flagged']:,} candidate flows -> {ing_ab['ml_only']['n_incidents']:,} incidents")
print(f"Hybrid (ML | R104) Ingestion: {ing_ab['ml_plus_r104']['n_raw_flagged']:,} candidate flows -> {ing_ab['ml_plus_r104']['n_incidents']:,} incidents")
print(f"Deterministic Rule Hits Added: +{ing_ab['delta_raw_flows']:,} flows")

## 6. Pre-Registration Merge Hypotheses Verification
Auditing cross-threat campaign merge predictions made in `preregistration_v1.md` before execution:

In [ ]:
prereg_verif = summary['preregistration_verification']
prereg_rows = []
labels_prereg = {
    'slowloris_to_Slowhttptest_300s': 'DoS slowloris -> DoS Slowhttptest (G=300s)',
    'hulk_to_goldeneye_300s': 'DoS Hulk -> DoS GoldenEye (G=300s)',
    'xss_to_sqli_300s': 'Web XSS -> SQL Injection (G=300s)',
    'slowhttptest_to_hulk_900s': 'DoS Slowhttptest -> DoS Hulk (G=900s)',
    'web_brute_force_to_xss_900s': 'Web Brute Force -> Web XSS (G=900s)',
}

for k, v in prereg_verif.items():
    prereg_rows.append({
        'Campaign Merge Hypothesis': labels_prereg.get(k, k),
        'Predicted': 'Merge' if v['predicted'] else 'Split',
        'Observed': 'Merged' if v['observed'] else 'Split',
        'Audit Status': 'CONFIRMED' if v['matches_preregistration'] else 'DISCREPANCY (empirical interval > 300s)'
    })

prereg_df = pd.DataFrame(prereg_rows)
display(prereg_df)